In [1]:
import networkx as nx
import numpy as np
import pandas as pd 
import scipy.stats
import random


from tqdm import tqdm

#for copy
import copy

In [2]:
def perform_contagion_transfers_weighted_version(graph, infected_list, mode='simple', threshold_complex=0.1, beta_simple=0.039,prob_complex=-1):
    """
    Simulate one step of contagion in a weighted graph.

    Parameters:
    graph (networkx.Graph): The weighted graph where the contagion takes place.
    infected_list (list): List of currently infected nodes.
    mode (str): The mode of contagion, either 'simple' or 'complex'.
    threshold_complex (float): The threshold for complex contagion.
    beta_simple (float): The infection probability factor for simple contagion.

    Returns:
    tuple: A tuple containing:
        - new_infected_marker (bool): True if new infections occurred, False otherwise.
        - infected_list (list): Updated list of infected nodes.
    """

    if mode == 'simple':
        new_infected_marker = False
        new_infected_nodes = []

        all_nodes = list(graph.nodes)

        for i in all_nodes:
            if i in infected_list:
                continue
            else:
                pr_infection = 0
                pi_calc = 1
                edges = list(graph.neighbors(i))
                for j in edges:
                    if j in infected_list:
                        # Update infection probability
                        pi_calc *= (1 - beta_simple * graph.get_edge_data(i, j)['weight'])
                        pr_infection = 1 - pi_calc

                # Determine if node i gets infected
                if random.uniform(0,1) <= pr_infection:
                    new_infected_marker = True
                    new_infected_nodes.append(i)

        # Update the infected list with new infections
        for i in new_infected_nodes:
            infected_list.append(i)

        return new_infected_marker, infected_list

    else:  # Complex contagion
        new_infected_marker = False
        new_infected_nodes = []

        all_nodes = list(graph.nodes)
        for i in all_nodes:
            if i in infected_list:
                continue
            else:
                summation_infected = 0
                summation_whole = 0
                edges = list(graph.neighbors(i))
                for j in edges:
                    # Sum the weights of all edges
                    summation_whole += graph.get_edge_data(i, j)['weight']
                    if j in infected_list:
                        # Sum the weights of edges to infected neighbors
                        summation_infected += graph.get_edge_data(i, j)['weight']

                if summation_whole == 0:
                    print('Something is wrong')
                else:
                    # Check if the proportion of infected neighbors exceeds the threshold
                    if summation_infected / summation_whole >= threshold_complex:
                        new_infected_marker = True
                        new_infected_nodes.append(i)
                    elif 0 < summation_infected and random.uniform(0,1) <= prob_complex:
                        new_infection_marker = True
                        new_infected_nodes.append(i)

        # Update the infected list with new infections
        for i in new_infected_nodes:
            infected_list.append(i)

        return new_infected_marker, infected_list



In [3]:
def perform_simple_contagion_weighted(beta_simple, graph, initially_infected):
    """
    Performs a simple contagion process on a graph and computes the corresponding
    extended persistent homology (EPH) based on the infection steps.

    Parameters:
    - beta_simple (float): The transmission probability for each infected neighbor in simple mode.
    - graph (networkx.Graph): The network graph over which the contagion spreads.
    - initially_infected (list): List of nodes that are initially infected.

    Returns:
    - list: A list of the mean lifetime of features in the computed extended persistent homology for each contagion step.
    - mean of the above List
    - rho (correlation of PRL article)

    Note:
    - The function iterates over the contagion process until all nodes are infected or no new infections occur.
    - It computes the EPH based on the order of infection of nodes, using the subgraph of infected nodes at each step.
    """
    # Initialize
    infected_list = initially_infected[:]
    time_current = 0
    new_infections = True
    step_of_infection_per_node = [None] * len(graph.nodes)
    simple_list_EPH = []

    # Mark initially infected nodes with their infection step (time 0)
    for node_index in initially_infected:
        node_pos = list(graph.nodes).index(node_index)
        step_of_infection_per_node[node_pos] = time_current

    # Perform contagion until all nodes are infected or no new infections
    while new_infections and len(infected_list) < len(graph.nodes):
        time_current += 1
        new_infections, infected_list = perform_contagion_transfers_weighted_version(
            graph, infected_list, mode='simple', beta_simple=beta_simple)

        # Update infection steps for newly infected nodes
        for node_index in infected_list:
            node_pos = list(graph.nodes).index(node_index)
            if step_of_infection_per_node[node_pos] is None:
                step_of_infection_per_node[node_pos] = time_current

    return np.array(step_of_infection_per_node)

In [4]:
def perform_complex_contagion_weighted(threshold, graph, initially_infected,prob_complex=-1):
    """
    Performs a complex contagion process on a graph and computes the corresponding
    extended persistent homology (EPH) based on the infection steps.

    Parameters:
    - threshold (int): The threshold number of infected neighbors required for a node to become infected.
    - graph (networkx.Graph): The network graph over which the contagion spreads.
    - initially_infected (list): List of nodes that are initially infected.
    - prob_complex (float): The probability of infection in complex mode if the number of infected neighbors is below the threshold but greater than 1.

    Returns:
    - list: A list of the mean lifetime of features in the computed extended persistent homology for each contagion step.

    Note:
    - The function iterates over the contagion process until all nodes are infected or no new infections occur.
    - It computes the EPH based on the order of infection of nodes, using the subgraph of infected nodes at each step.
    """
    # Initialize
    infected_list = initially_infected
    time_current = 0
    new_infections = True
    step_of_infection_per_node = [None] * len(graph.nodes)
    complex_list_EPH = []

    # Mark initially infected nodes with their infection step (time 0)
    for node_index in initially_infected:
        node_pos = list(graph.nodes).index(node_index)
        step_of_infection_per_node[node_pos] = time_current

    # Perform contagion until all nodes are infected or no new infections
    while new_infections and len(infected_list) < len(graph.nodes):
        time_current += 1
        new_infections, infected_list = perform_contagion_transfers_weighted_version(
            graph, infected_list, mode='complex', threshold_complex=threshold,prob_complex=prob_complex)

        # Update infection steps for newly infected nodes
        for node_index in infected_list:
            node_pos = list(graph.nodes).index(node_index)
            if step_of_infection_per_node[node_pos] is None:
                step_of_infection_per_node[node_pos] = time_current
    
    return np.array(step_of_infection_per_node)


In [5]:
# betha_set = [0.6,0.7,0.8]
# betha_set = [0.1,0.2,0.3,0.4,0.5]
betha_set = [0.01,0.1,0.2,0.3,0.4,0.5,0.6]


In [6]:
def init_nodes_var_reduction(iter_cnt,arr,k):
    rng = np.random.RandomState(iter_cnt)
    init_nodes = rng.choice(arr,k)
    return list(init_nodes)

In [7]:
network_list = [
    'conf','email_eu_modified','hospital','school','work'
]

In [8]:
for network_name in network_list:
    
    #reading graph 
    G = nx.read_graphml(f'../networks/weighted/G_weighted_{network_name}.graphml')
    
    #preprocessing
    # Create a mapping from current node names (str) to integers
    mapping = {node: int(float(node)) for node in G.nodes()}
    # Relabel the nodes in the graph using the mapping
    G = nx.relabel_nodes(G, mapping)


    #weight normalization 
    # Step 1: Extract all weights
    weights = [data['weight'] for u, v, data in G.edges(data=True) if 'weight' in data]

    # Step 2: Find the maximum weight
    max_weight = max(weights)

    # Step 3: Normalize the weights
    for u, v, data in G.edges(data=True):
        if 'weight' in data:
            data['weight'] = data['weight'] / max_weight
            
            
    #storing simulations
    order_collection_simulations = [] 
    seed_sim = [] 
    betha_sim = [] 
    
    #doing simulations
    for beta in tqdm(betha_set):
        for sample_it in range(300):
            #initilize graph and infected nodes
            # beta = init_nodes_var_reduction(iter_cnt=sample_it,arr=betha_set,k=1)[0]
    
            list_inf = init_nodes_var_reduction(iter_cnt=sample_it,
                                                arr=list(G.nodes),k=10)
            order = perform_simple_contagion_weighted(beta_simple=beta, graph=copy.deepcopy(G), 
                                                            initially_infected=copy.deepcopy(list_inf))
    
            order_collection_simulations.append(order)
            seed_sim.append(sample_it)
            betha_sim.append(beta)


    
    result = pd.DataFrame(order_collection_simulations)
    result['betha'] = betha_sim
    result['seed'] = seed_sim

    #saving 
    result.to_csv(f'../results/weighted/given_betha_n=300/weighted_simple_{network_name}.csv')    

100%|██████████| 7/7 [00:04<00:00,  1.63it/s]
